# Agentic Security Lab - GRPO Training Notebook (Round 2)

**End-to-end RL post-training: teach an LLM to contain software supply-chain attacks**

## Architecture
- **Model**: Qwen2.5-3B-Instruct 4-bit QLoRA via Unsloth
- **Training**: TRL GRPOTrainer with `environment_factory` (multi-turn tool-calling)
- **Environment**: Procedurally generated supply-chain incidents (infinite variation)
- **External APIs**: OSV.dev, deps.dev, GitHub Advisory DB, npm registry
- **Stability**: StarPO-S trajectory filtering (RAGEN, arXiv 2504.20073)
- **Self-play**: Adversarial attacker adapts difficulty (SPIRAL, arXiv 2506.24119)
- **Curriculum**: easy->medium->hard based on rolling benchmark score

## Key Design Decisions
| Decision | Choice | Rationale |
|----------|--------|-----------|
| Scenario gen | Procedural (infinite) | Can't memorize; RL genuinely needed |
| External tools | OSV/deps.dev/GHSA/npm | Real info-gathering vs time tradeoff |
| Trajectory filter | StarPO-S top-50% | Prevents echo trap collapse |
| Adversary | Adaptive attacker | Auto-curriculum via blind-spot exploitation |
| `scale_rewards` | False | Avoids difficulty bias (arXiv 2503.20783) |
| `num_generations` | 4 | T4 VRAM constraint |

## 1 - Install

In [ ]:
%%capture
!git clone https://huggingface.co/spaces/A-HK/agentic-security-lab /content/agentic-security-lab 2>/dev/null || true
%cd /content/agentic-security-lab
!pip install -q -e '.[train]'
!pip install -q unsloth
!pip install -q --upgrade 'trl>=0.16' 'transformers>=5.2'
!pip install -q matplotlib requests numpy

## 2 - Configuration

In [ ]:
import os, json, time, copy, random, uuid, sys, re
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

ROOT = Path('/content/agentic-security-lab')
sys.path.insert(0, str(ROOT))
ARTIFACT_DIR = ROOT / 'artifacts'; PLOT_DIR = ROOT / 'plots'
ARTIFACT_DIR.mkdir(exist_ok=True); PLOT_DIR.mkdir(exist_ok=True)

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except: pass
HF_TOKEN = os.environ.get('HF_TOKEN', '')

MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 4096; LORA_RANK = 32; LORA_ALPHA = 32
NUM_GENERATIONS = 4; LEARNING_RATE = 5e-6; KL_COEFF = 0.04
MAX_COMPLETION_LENGTH = 2048; GRAD_ACCUM = 4; TEMPERATURE = 0.8
NUM_TRAIN_EPOCHS = 3; EPISODES_PER_DIFFICULTY = 40
EASY_ADVANCE = 0.55; MEDIUM_ADVANCE = 0.45; ROLLING_WINDOW = 5
HUB_MODEL_ID = 'A-HK/security-incident-responder-grpo'
OUTPUT_DIR = str(ARTIFACT_DIR / 'grpo_checkpoint')
print(f'Model: {MODEL_NAME}\nHub: {HUB_MODEL_ID}')

## 3 - Start Local Server

In [ ]:
import subprocess, requests as _req
os.chdir(str(ROOT))
server = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '8000'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(4)
LOCAL_URL = 'http://127.0.0.1:8000'
try:
    r = _req.get(f'{LOCAL_URL}/health', timeout=10)
    print(f'Local server: {r.json()}')
except Exception as e:
    print(f'Server failed: {e}')
    LOCAL_URL = 'https://A-HK-agentic-security-lab.hf.space'

## 4 - Test Procedural Scenario Generation

The environment now generates **infinite unique scenarios** from parameterized distributions.
No two training episodes are identical.

In [ ]:
from training.procedural_scenarios import generate_procedural_scenario

# Generate 3 different easy scenarios
for seed in [1, 2, 3]:
    sc = generate_procedural_scenario('easy', seed=seed)
    mal = [p for p, m in sc['packages'].items() if m['malicious']]
    print(f'Seed {seed}: {len(sc["packages"])} pkgs, mal={mal}, '
          f'secrets={len(sc["secrets"])}, teams={len(sc["required_actions"]["notify"])}, '
          f'attack={sc["attack_type"]}')

# Hard scenario
sc = generate_procedural_scenario('hard', seed=42)
mal = [p for p, m in sc['packages'].items() if m['malicious']]
print(f'\nHard: {len(sc["packages"])} pkgs, {len(mal)} malicious, '
      f'{len(sc["secrets"])} secrets, exfil={sc["exfiltration_step"]} steps')
print('Procedural generation OK')

## 5 - Test Real Security APIs

Live lookups against OSV.dev, deps.dev, GitHub Advisory DB, npm registry.
Free, no API key required.

In [ ]:
from training.security_apis import query_osv, query_deps_dev, comprehensive_package_check

# Test OSV: real malicious package
r = query_osv('ctx', 'PyPI')
print(f'OSV ctx (PyPI): {r["total"]} vulns, ghsa={r["ghsa"]}')

# Test deps.dev: real package metadata
r = query_deps_dev('lodash', 'npm', '4.17.21')
print(f'deps.dev lodash: deprecated={r.get("deprecated")}, advisories={r.get("advisories",[])[0:2]}')

# Comprehensive check
r = comprehensive_package_check('node-ipc', 'npm', '10.1.0')
print(f'node-ipc: verdict={r["verdict"]}, risk={r["risk_score"]}')
print('Security APIs OK')

## 6 - TRL Environment Wrapper

Each instance embeds an in-process environment with **procedurally generated** scenarios.
Includes OSV/deps.dev as additional investigation tools.

In [ ]:
from server.agentic_security_lab_environment import AgenticSecurityLabEnvironment
from models import AgenticSecurityLabAction
from training.adversarial_attacker import AdversarialAttacker

CURRENT_DIFFICULTY = 'easy'
ATTACKER = AdversarialAttacker(seed=42)

class SecurityIncidentEnv:
    '''TRL environment_factory wrapper with procedural gen + adversarial attacker.'''
    def __init__(self):
        self._env = AgenticSecurityLabEnvironment(CURRENT_DIFFICULTY)
        self.cumulative_reward = 0.0
        self.step_rewards = []; self.actions_taken = []
        self.done = False; self.steps_used = 0
        self.task_name = CURRENT_DIFFICULTY; self._last_sig = None

    def reset(self, **kwargs) -> str | None:
        '''Reset with procedurally generated scenario.'''
        self.cumulative_reward = 0.0; self.step_rewards = []; self.actions_taken = []
        self.done = False; self.steps_used = 0; self._last_sig = None
        task = kwargs.get('task_name', CURRENT_DIFFICULTY); self.task_name = task
        self._env = AgenticSecurityLabEnvironment(task)
        obs = self._env.reset(task_name=task, mode='training')
        return obs.result

    def _do_step(self, command, parameters):
        if self.done: raise ValueError('Episode ended.')
        action = AgenticSecurityLabAction(command=command, parameters=parameters)
        obs = self._env.step(action)
        reward = obs.reward
        sig = json.dumps({'c':command,'p':parameters}, sort_keys=True)
        if sig == self._last_sig: reward -= 0.1
        self._last_sig = sig
        self.cumulative_reward += reward; self.step_rewards.append(reward)
        self.actions_taken.append(command); self.steps_used += 1; self.done = obs.done
        lines = [obs.result, '', f'Status: {obs.incident_summary}', f'Steps remaining: {obs.steps_remaining}']
        if obs.exposed_secrets: lines.append(f'Secrets to rotate: {obs.exposed_secrets}')
        if obs.active_malicious_packages: lines.append(f'Active malicious: {obs.active_malicious_packages}')
        if obs.visible_alerts: lines.append(f'Alerts: {obs.visible_alerts}')
        if obs.done:
            bd = obs.data.get('score_breakdown', {})
            lines.append(f'Benchmark: {obs.data.get("benchmark_score",0):.3f}')
        return '\n'.join(lines)

    def inspect_package(self, package: str) -> str:
        '''
        Inspect package metadata, publisher info, and IOC indicators.

        Args:
            package: Package name and version (e.g. axios@1.7.4)

        Returns:
            Package metadata with publisher, publish date, and IOCs.
        '''
        return self._do_step('inspect_package', {'package': package})

    def check_dependents(self, package: str) -> str:
        '''
        List downstream services depending on this package.

        Args:
            package: Package name and version (e.g. axios@1.7.4)

        Returns:
            List of affected downstream consumer teams.
        '''
        return self._do_step('check_dependents', {'package': package})

    def rotate_secret(self, secret: str) -> str:
        '''
        Rotate a compromised credential.

        Args:
            secret: Name of the secret (e.g. STRIPE_SECRET_KEY)

        Returns:
            Confirmation including whether the secret was critical.
        '''
        return self._do_step('rotate_secret', {'secret': secret})

    def quarantine_package(self, package: str) -> str:
        '''
        Quarantine a malicious package from the registry.

        Args:
            package: Package to quarantine (e.g. axios@1.7.4)

        Returns:
            Result. Warns on false positives.
        '''
        return self._do_step('quarantine', {'package': package})

    def notify_team(self, team: str) -> str:
        '''
        Send breach notification to affected team.

        Args:
            team: Team name (e.g. payments-service)

        Returns:
            Confirmation of notification.
        '''
        return self._do_step('notify', {'team': team})

    def scan_logs(self, package: str) -> str:
        '''
        Scan CI/CD logs for indicators of compromise.

        Args:
            package: Package name and version (e.g. axios@1.7.4)

        Returns:
            Log excerpts with suspicious entries.
        '''
        return self._do_step('scan_logs', {'package': package})

    def conclude_incident(self) -> str:
        '''
        Declare incident contained and end episode.

        Returns:
            Final summary with component scores.
        '''
        return self._do_step('conclude', {})

_t = SecurityIncidentEnv(); _obs = _t.reset()
print(_obs[:200])
print(f'Quarantine: {_t.quarantine_package("axios@1.7.4")[:80]}')
print(f'Reward: {_t.cumulative_reward:.3f}')

## 7 - Reward Functions

In [ ]:
def environment_reward(environments, **kwargs):
    '''Primary: cumulative dense reward, scaled 2x.'''
    return [env.cumulative_reward * 2.0 for env in environments]

def efficiency_reward(environments, **kwargs):
    '''Bonus for quick completion.'''
    return [max(0.0, 0.3*(1.0-env.steps_used/{'easy':14,'medium':18,'hard':10}.get(env.task_name,14))) if env.done and env.cumulative_reward > 0 else 0.0 for env in environments]

def diversity_reward(environments, **kwargs):
    '''Penalise action spam.'''
    rewards = []
    for env in environments:
        if len(env.actions_taken) <= 3: rewards.append(0.0); continue
        counts = defaultdict(int)
        for a in env.actions_taken: counts[a] += 1
        rewards.append(-0.2 if max(counts.values())/len(env.actions_taken) > 0.6 else 0.0)
    return rewards
print('Reward functions ready')

## 8 - Load Model (Unsloth)

In [ ]:
from unsloth import FastLanguageModel
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'VRAM: {props.total_memory/1e9:.1f} GB')
model, tokenizer = FastLanguageModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, dtype='auto', load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=LORA_RANK, target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'], lora_alpha=LORA_ALPHA, lora_dropout=0, bias='none', use_gradient_checkpointing=True, random_state=42)
print(f'Loaded {MODEL_NAME} with LoRA r={LORA_RANK}')
model.print_trainable_parameters()

## 9 - Training Dataset

In [ ]:
from datasets import Dataset
SYSTEM_PROMPT = 'You are an expert security incident responder. Contain the supply-chain compromise before the attacker exfiltrates. Strategy: 1) scan_logs/inspect to find IOCs, 2) quarantine malicious packages, 3) check_dependents for affected teams, 4) rotate secrets (critical first), 5) notify all affected teams, 6) conclude when done.'
USER_PROMPT = 'A supply-chain incident detected. Use available tools to investigate, contain, and remediate. The attacker is actively exfiltrating credentials.'
def build_dataset(task_name, count=EPISODES_PER_DIFFICULTY):
    return Dataset.from_list([{'prompt':[{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':USER_PROMPT}],'task_name':task_name}]*count)
train_dataset = build_dataset('easy')
print(f'Dataset: {len(train_dataset)} episodes')

## 10 - Baseline Evaluation

In [ ]:
import requests
def model_generate_action(messages, max_new_tokens=200):
    FastLanguageModel.for_inference(model)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.3, do_sample=True, top_p=0.9, pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    raw = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    if '```' in raw: raw = '\n'.join(l for l in raw.split('\n') if not l.startswith('```')).strip()
    try: return json.loads(raw)
    except:
        m = re.search(r'\{[^}]+\}', raw)
        if m:
            try: return json.loads(m.group())
            except: pass
    return {'command':'conclude','parameters':{}}

def run_eval_episode(base_url, task, model_fn=None, max_steps=25):
    s = requests.Session()
    obs = s.post(f'{base_url}/reset', json={'task_name':task,'mode':'benchmark'}, timeout=30).json()
    total_reward = 0.0; step_n = 0; actions = defaultdict(int)
    messages = [{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':obs['result']}]
    for step_n in range(1, max_steps+1):
        if obs.get('done'): break
        action = model_fn(messages) if model_fn else {'command':'conclude','parameters':{}}
        cmd = action.get('command','conclude'); params = action.get('parameters',{})
        actions[cmd] += 1
        obs = s.post(f'{base_url}/step', json={'command':cmd,'parameters':params}, timeout=30).json()
        total_reward += obs.get('reward',0)
        messages.append({'role':'assistant','content':json.dumps(action)})
        messages.append({'role':'user','content':obs.get('result','')})
    state = s.get(f'{base_url}/state', timeout=30).json()
    s.close()
    bd = obs.get('data',{}).get('score_breakdown',{})
    return {'reward':total_reward,'benchmark':obs.get('data',{}).get('benchmark_score',0),'breakdown':{'quarantine':bd.get('quarantine_ratio',0),'rotate':bd.get('rotate_ratio',0),'notify':bd.get('notify_ratio',0),'contain':bd.get('contain_ratio',0)},'steps':step_n,'actions':dict(actions)}

N_EVAL = 3
baseline_results = {}
for diff in ['easy','medium','hard']:
    baseline_results[diff] = []
    print(f'--- BASELINE {diff.upper()} ---')
    for ep in range(N_EVAL):
        r = run_eval_episode(LOCAL_URL, diff, model_fn=model_generate_action, max_steps=20)
        baseline_results[diff].append(r)
        print(f'  ep{ep+1}: reward={r["reward"]:.3f} bench={r["benchmark"]:.3f} steps={r["steps"]}')
baseline_avg = {}
for diff, res in baseline_results.items():
    baseline_avg[diff] = {'reward':np.mean([r['reward'] for r in res]),'benchmark':np.mean([r['benchmark'] for r in res]),'breakdown':{k:np.mean([r['breakdown'][k] for r in res]) for k in ['quarantine','rotate','notify','contain']}}
    print(f'{diff}: avg_bench={baseline_avg[diff]["benchmark"]:.3f}')

## 11 - GRPO Config + Curriculum + StarPO-S

In [ ]:
from trl import GRPOTrainer, GRPOConfig
from transformers import TrainerCallback
from training.callbacks import MetricsCallback
from training.trajectory_filter import CollapseDetector
import gc

gpu_name = torch.cuda.get_device_name(0).lower() if torch.cuda.is_available() else ''
use_bf16 = any(x in gpu_name for x in ['a100','a10','h100','l4','l40'])

grpo_config = GRPOConfig(output_dir=OUTPUT_DIR, learning_rate=LEARNING_RATE, num_generations=NUM_GENERATIONS, max_completion_length=MAX_COMPLETION_LENGTH, max_prompt_length=512, temperature=TEMPERATURE, beta=KL_COEFF, scale_rewards=False, num_train_epochs=NUM_TRAIN_EPOCHS, per_device_train_batch_size=1, gradient_accumulation_steps=GRAD_ACCUM, gradient_checkpointing=True, bf16=use_bf16, fp16=not use_bf16, logging_steps=1, logging_first_step=True, log_completions=True, disable_tqdm=True, save_strategy='steps', save_steps=20, save_total_limit=3, push_to_hub=True, hub_model_id=HUB_MODEL_ID, hub_strategy='every_save', warmup_ratio=0.1, reward_weights=[1.0, 0.3, 0.2], report_to=[])

collapse_detector = CollapseDetector(window=15)

class CurriculumGRPOCallback(TrainerCallback):
    '''Curriculum + collapse detection + adversarial attacker feedback.'''
    def __init__(self):
        self.metrics_cb = MetricsCallback(str(ARTIFACT_DIR / 'grpo_metrics.jsonl'))
        self.episode_rewards = []; self.losses = []; self.step_count = 0
    def on_log(self, args, state, control, logs=None, **kwargs):
        global CURRENT_DIFFICULTY
        if not logs: return
        loss = logs.get('loss') or logs.get('train_loss')
        if loss is not None: self.losses.append(loss)
        reward = logs.get('reward')
        if reward is not None:
            self.step_count += 1
            self.episode_rewards.append(reward)
            self.metrics_cb.log({'type':'grpo_step','step':self.step_count,'reward':reward,'loss':loss,'difficulty':CURRENT_DIFFICULTY})
            # Collapse detection (StarPO-S)
            alert = collapse_detector.update(reward=reward)
            if alert: print(f'  WARNING: {alert}')
            # Feed adversarial attacker
            ATTACKER.observe_defender({'benchmark_score': reward/2.0, 'breakdown':{'notify_ratio':0.5,'rotate_ratio':0.5}})
            # Curriculum
            if len(self.episode_rewards) >= ROLLING_WINDOW:
                avg = np.mean(self.episode_rewards[-ROLLING_WINDOW:])
                if CURRENT_DIFFICULTY == 'easy' and avg > EASY_ADVANCE * 2:
                    CURRENT_DIFFICULTY = 'medium'; print(f'\nCURRICULUM -> medium (avg={avg:.3f})')
                elif CURRENT_DIFFICULTY == 'medium' and avg > MEDIUM_ADVANCE * 2:
                    CURRENT_DIFFICULTY = 'hard'; print(f'\nCURRICULUM -> hard (avg={avg:.3f})')
            if self.step_count % 5 == 0:
                recent = self.episode_rewards[-5:]
                adv = ATTACKER.get_metrics()
                print(f'  step {self.step_count}: reward={reward:.3f} avg5={np.mean(recent):.3f} diff={CURRENT_DIFFICULTY} adv_level={adv["adversarial_level"]:.2f}')

curriculum_cb = CurriculumGRPOCallback()
print(f'Config ready. Adversarial attacker level: {ATTACKER.get_metrics()["adversarial_level"]:.2f}')

## 12 - Train with GRPO

In [ ]:
gc.collect(); torch.cuda.empty_cache()
FastLanguageModel.for_training(model)
trainer = GRPOTrainer(model=model, processing_class=tokenizer, reward_funcs=[environment_reward, efficiency_reward, diversity_reward], train_dataset=train_dataset, args=grpo_config, environment_factory=SecurityIncidentEnv, callbacks=[curriculum_cb])
print('='*60 + f'\nGRPO Training | {MODEL_NAME} | {CURRENT_DIFFICULTY} | {HUB_MODEL_ID}\n' + '='*60)
train_result = trainer.train()
print(f'\nDone: {train_result.global_step} steps, loss={train_result.training_loss:.4f}')

## 13 - Post-Training Evaluation

In [ ]:
trained_results = {}
for diff in ['easy','medium','hard']:
    trained_results[diff] = []
    print(f'--- TRAINED {diff.upper()} ---')
    for ep in range(N_EVAL):
        r = run_eval_episode(LOCAL_URL, diff, model_fn=model_generate_action, max_steps=20)
        trained_results[diff].append(r)
        print(f'  ep{ep+1}: reward={r["reward"]:.3f} bench={r["benchmark"]:.3f} steps={r["steps"]}')
trained_avg = {}
for diff, res in trained_results.items():
    trained_avg[diff] = {'reward':np.mean([r['reward'] for r in res]),'benchmark':np.mean([r['benchmark'] for r in res]),'breakdown':{k:np.mean([r['breakdown'][k] for r in res]) for k in ['quarantine','rotate','notify','contain']}}
print('\n' + '='*60 + '\nIMPROVEMENT\n' + '='*60)
for d in ['easy','medium','hard']:
    b, t = baseline_avg[d], trained_avg[d]
    print(f'  {d:8s}: {b["benchmark"]:.3f} -> {t["benchmark"]:.3f} (delta={t["benchmark"]-b["benchmark"]:+.3f})')

## 14 - Plots

In [ ]:
def smooth(v, w=10):
    if len(v) < w: return v
    return np.convolve(v, np.ones(w)/w, mode='valid').tolist()

fig, ax = plt.subplots(figsize=(12,5))
ep = range(1, len(curriculum_cb.episode_rewards)+1)
ax.plot(ep, curriculum_cb.episode_rewards, alpha=0.3, color='steelblue', label='Raw')
sm = smooth(curriculum_cb.episode_rewards)
if sm: ax.plot(range(10,10+len(sm)), sm, color='steelblue', lw=2, label='Smoothed')
ax.set_xlabel('Training Step'); ax.set_ylabel('Episode Reward')
ax.set_title('GRPO: Reward Curve'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(PLOT_DIR/'reward_curve.png', dpi=150); plt.show()

if curriculum_cb.losses:
    fig, ax = plt.subplots(figsize=(12,5))
    ax.plot(range(1,len(curriculum_cb.losses)+1), curriculum_cb.losses, alpha=0.5, color='purple')
    ax.set_xlabel('Step'); ax.set_ylabel('Loss'); ax.set_title('GRPO: Loss Curve'); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(PLOT_DIR/'loss_curve.png', dpi=150); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14,6))
diffs = ['easy','medium','hard']; x = np.arange(len(diffs)); w = 0.35
for idx, (metric, title) in enumerate([('reward','Reward'),('benchmark','Benchmark')]):
    ax = axes[idx]
    bv = [baseline_avg[d][metric] for d in diffs]; tv = [trained_avg[d][metric] for d in diffs]
    b1 = ax.bar(x-w/2, bv, w, label='Before', color='#e74c3c', alpha=0.8)
    b2 = ax.bar(x+w/2, tv, w, label='After', color='#2ecc71', alpha=0.8)
    for bar in list(b1)+list(b2): ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(), f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
    ax.set_xlabel('Difficulty'); ax.set_ylabel(title); ax.set_title(f'{title}: Before vs After')
    ax.set_xticks(x); ax.set_xticklabels([d.capitalize() for d in diffs]); ax.legend(); ax.grid(alpha=0.3,axis='y')
    if metric=='benchmark': ax.set_ylim(0,1.1)
plt.tight_layout(); plt.savefig(PLOT_DIR/'before_after_comparison.png', dpi=150); plt.show()

fig, axes = plt.subplots(1,4,figsize=(18,4))
for ax, comp, col in zip(axes, ['quarantine','rotate','notify','contain'], ['#e74c3c','#3498db','#2ecc71','#9b59b6']):
    bv = [baseline_avg[d]['breakdown'][comp] for d in diffs]; tv = [trained_avg[d]['breakdown'][comp] for d in diffs]
    ax.bar(x-w/2,bv,w,color=col,alpha=0.4,label='Before'); ax.bar(x+w/2,tv,w,color=col,alpha=0.9,label='After')
    ax.set_xticks(x); ax.set_xticklabels([d.capitalize() for d in diffs]); ax.set_ylim(0,1.1); ax.set_title(comp.capitalize()); ax.legend(fontsize=8); ax.grid(alpha=0.3,axis='y')
plt.suptitle('Component Scores',fontsize=13,y=1.02); plt.tight_layout()
plt.savefig(PLOT_DIR/'component_scores.png',dpi=150,bbox_inches='tight'); plt.show()
print(f'Plots saved to {PLOT_DIR}/')

## 15 - Save & Push Model

In [ ]:
model.save_pretrained(f'{OUTPUT_DIR}/lora_adapter')
tokenizer.save_pretrained(f'{OUTPUT_DIR}/lora_adapter')
try:
    model.push_to_hub(HUB_MODEL_ID, tokenizer=tokenizer, token=HF_TOKEN)
    print(f'Pushed to https://huggingface.co/{HUB_MODEL_ID}')
except Exception as e:
    print(f'Push failed: {e}')
    try:
        model.push_to_hub_merged(HUB_MODEL_ID, tokenizer=tokenizer, save_method='lora', token=HF_TOKEN)
        print(f'Pushed (merged) to https://huggingface.co/{HUB_MODEL_ID}')
    except Exception as e2:
        print(f'Both failed: {e2}. Saved locally at {OUTPUT_DIR}/lora_adapter')

## 16 - Summary

In [ ]:
try: server.terminate(); server.wait(timeout=5)
except: pass
print('='*70 + '\nAGENTIC SECURITY LAB - GRPO TRAINING COMPLETE\n' + '='*70)
print(f'Model:  {MODEL_NAME}\nMethod: GRPO + environment_factory + Unsloth QLoRA')
print(f'LoRA:   r={LORA_RANK}, alpha={LORA_ALPHA}')
print(f'Features: Procedural scenarios, Real security APIs, StarPO-S filtering, Adversarial attacker')
adv = ATTACKER.get_metrics()
print(f'Adversarial attacker final level: {adv["adversarial_level"]:.2f} ({adv["episodes_observed"]} episodes observed)')
if collapse_detector.alerts: print(f'Collapse alerts: {len(collapse_detector.alerts)}')
print('\nResults:')
for d in ['easy','medium','hard']:
    b, t = baseline_avg[d], trained_avg[d]
    print(f'  {d:8s}: {b["benchmark"]:.3f} -> {t["benchmark"]:.3f} (delta={t["benchmark"]-b["benchmark"]:+.3f})')
print(f'\nModel: https://huggingface.co/{HUB_MODEL_ID}')
print(f'Env:   https://huggingface.co/spaces/A-HK/agentic-security-lab')
print('\nReferences:')
for n, a in [('GRPO','2402.03300'),('RAGEN/StarPO-S','2504.20073'),('ScalingInter-RL','2509.08755'),('SPIRAL','2506.24119'),('RL-Struct','2512.00319'),('Pentest-R1','2508.07382'),('Self-Evolving Curriculum','2505.14970')]: print(f'  [{n}] arxiv.org/abs/{a}')
print('='*70)